# Model Comparison Benchmark

Membandingkan Qwen2.5-3B-Instruct, model SFT, dan model GRPO pada 60 kasus serta artefak retrieval final yang sama. Runner mencatat valid-output rate, citation precision, abstention accuracy, latency generation, peak VRAM, dan loaded model footprint. Faithfulness serta answer relevance tetap memerlukan audit manual atas predictions sebelum model production dipilih.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
BRANCH = "main"

%cd /content
!test -d indonesian-legal-compliance-rag || git clone --depth 1 --branch {BRANCH} https://github.com/FadhilahAfif/indonesian-legal-compliance-rag.git
!git -C indonesian-legal-compliance-rag checkout {BRANCH}
!git -C indonesian-legal-compliance-rag pull --ff-only origin {BRANCH}
%cd /content/indonesian-legal-compliance-rag
!git rev-parse --short HEAD

In [ ]:
!python -m pip install -q -r requirements.txt
!python -m pip uninstall -y -q torchvision torchcodec
!python scripts/check_environment.py
!python -m unittest discover -s tests -v
!python -m eval.validate_cases --require-reviewed

In [ ]:
!python -m eval.compare_models

In [ ]:
import json
from pathlib import Path

comparison = json.loads(
    Path("eval/results/model-comparison/comparison.json").read_text(encoding="utf-8")
)
{
    label: {
        "model": report["config"]["model"],
        "model_stats": report["model"],
        "generation": report["metrics"]["generation"],
        "safety": report["metrics"]["safety"],
        "runtime": report["metrics"]["runtime"],
    }
    for label, report in comparison["models"].items()
}

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive(
    "model-comparison-results", "zip", "eval/results/model-comparison"
)
files.download(archive)